# Test Telegram Connection for TradingV1
Verify connection to Telegram using TelegramClientHandler and ConfigManager.

In [1]:
import sys
import os
import asyncio

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
print(f'Project root added to path: {project_root}')

from helper.telegram_client import TelegramClientHandler
from helper.config_manager import ConfigManager
from helper.Logger import Logging

# Initialize config and logger
try:
    print('Loading ConfigManager...')
    config = ConfigManager(config_file='../config/combined_config.yaml')
    print('ConfigManager loaded')
    # Override logging config to use project root
    if 'logging' not in config._data:
        config._data['logging'] = {}
    config._data['logging']['log_dir_name'] = os.path.join(project_root, 'logs')
    logger = Logging(config=config, instance_id='test_telegram')
    logger.set_log_file('test_telegram')
    print('Logger initialized')
except Exception as e:
    print(f'Failed to initialize config or logger: {e}')
    raise

# Get Telegram config
try:
    print('Fetching telegram config...')
    tele_config = config.get_section('telegram')
    print(f'Telegram config before modification: {tele_config}')
    # Adjust session_path to be relative to project root
    tele_config['session_path'] = os.path.join(project_root, tele_config['session_path'])
    print(f'Telegram config after modification: {tele_config}')
except KeyError as e:
    print(f'KeyError accessing telegram config: {e}')
    raise

# Verify session file
session_path = tele_config['session_path']
if not os.path.exists(session_path):
    print(f'Session file missing: {session_path}')
    raise FileNotFoundError(session_path)
print(f'Session file exists: {session_path}')

# Test Telegram connection
async def test_connection():
    handler = TelegramClientHandler(config, logger)
    try:
        logger.info('Attempting to connect to Telegram...')
        client = await handler.get_client()
        logger.info('Successfully connected to Telegram!')
        dialogs = await client.get_dialogs(limit=5)
        print(f'Fetched {len(dialogs)} dialogs: {[d.name for d in dialogs]}')
    except Exception as e:
        logger.error(f'Failed to connect to Telegram: {e}')
        raise
    finally:
        await handler.disconnect()
        logger.close_log()

# Run async test
try:
    print('Running Telegram connection test...')
    loop = asyncio.get_event_loop()
    if loop.is_running():
        task = loop.create_task(test_connection())
        await task
    else:
        loop.run_until_complete(test_connection())
except Exception as e:
    print(f'Error running test_connection: {e}')

Project root added to path: P:\DynaPOD\proj\trader


MyLogger.test_telegram - INFO - Attempting to connect to Telegram...


Loading ConfigManager...
ConfigManager loaded
Logger initialized
Fetching telegram config...
Telegram config before modification: {'api_id': 27715324, 'api_hash': '72f0a5168ab258f7b6cb169c78ff31d6', 'phone': '+14698380038', 'session_path': 'sessions/my_telegram_client.session'}
Telegram config after modification: {'api_id': 27715324, 'api_hash': '72f0a5168ab258f7b6cb169c78ff31d6', 'phone': '+14698380038', 'session_path': 'P:\\DynaPOD\\proj\\trader\\sessions/my_telegram_client.session'}
Session file exists: P:\DynaPOD\proj\trader\sessions/my_telegram_client.session
Running Telegram connection test...


MyLogger.test_telegram - INFO - Telegram connection established for phone 0038.
MyLogger.test_telegram - INFO - Successfully connected to Telegram!
MyLogger.test_telegram - INFO - Telegram connection closed.


Fetched 5 dialogs: ['CryptoS|Trade | Trade Room', 'White Rose Crypto', 'UXLINK® 2', 'CRYPTO WORLD DISCUSSION', 'CRYPTO WORLD UPDATES']
